# Pregunta 2 — Agrupamiento de elaboraciones similares

**Pregunta de investigación:** ¿Se pueden encontrar grupos de elaboraciones con consumos y composición de leche similares y rendimientos diferentes?

**Fuente:** `registro_produccion_LIMPIO (4).xlsx`, hoja `Registro Produccion`. Se utiliza el archivo original **solo en lectura**. Los precios ya fueron divididos por 5 en origen y **no se vuelven a dividir**.

**Objetivo del análisis:** Detectar grupos de producciones por consumos estandarizados y composición, comparando sus rendimientos medios.

**Indicador:** Indicadores calculados en las celdas siguientes, con unidad y período declarados.


## 1. Contexto
Dos elaboraciones pueden utilizar cantidades distintas de insumos aun cuando procesan volúmenes parecidos.

## 2. Gancho
**¿Qué puede sorprender?** La diferencia entre elaboraciones, grupos o períodos podría ser mayor de lo que sugiere mirar únicamente un promedio general. La magnitud exacta se establece con los datos, no se supone.

## 3. Preparación reproducible
Las transformaciones se calculan en memoria. No se modifica ninguna celda del Excel original.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
from artifact_tool import Blob, SpreadsheetFile
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, r2_score, accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
SEMILLA=42
ruta=Path('registro_produccion_LIMPIO (4).xlsx')
if not ruta.exists(): ruta=Path('/mnt/data/registro_produccion_LIMPIO (4).xlsx')
if not ruta.exists(): raise FileNotFoundError('Colocá el Excel junto al notebook')
w=SpreadsheetFile.import_xlsx(Blob.load(str(ruta)))
mat=w.worksheets.get_item_at(0).get_range('A1:CA1112').values
headers=mat[0]; registros=mat[1:]; ix={h:i for i,h in enumerate(headers)}
def valor(r,col):
    x=r[ix[col]]
    try: return float(x) if x is not None and x!='' else np.nan
    except (ValueError,TypeError): return np.nan
def matriz(cols,filas=registros): return np.array([[valor(r,c) for c in cols] for r in filas],float)
def tasas(r):
    l=valor(r,'LITROS DE LECHE')
    return [valor(r,c)*1000/l if np.isfinite(l) and l>0 else np.nan for c in ['AZUCAR','SUERO','BICARBONATO','L.P.D']]
def features():return np.array([[valor(r,'LITROS DE LECHE'),*tasas(r),valor(r,'MATERIA GRASA'),valor(r,'PROTEINAS')] for r in registros],float)
X=features();masas=np.array([str(r[ix['MASA']]) for r in registros]);rend=matriz(['RENDIMIENTO %']).ravel();kg=matriz(['TOTAL KG OBTENIDOS']).ravel()
def normalizar(X):return Pipeline([('imputar',SimpleImputer(strategy='median')),('escalar',StandardScaler())]).fit_transform(X)
def grafico(titulo,unidad,transformacion):
    plt.title(titulo,loc='left',fontweight='bold',fontsize=13)
    plt.figtext(.01,.005,f'Fuente: Registro Produccion | {unidad} | {transformacion} | N={len(registros)}',fontsize=8,color='dimgray')
    plt.tight_layout(rect=(0,.055,1,1));plt.show()
print('Archivo leído en modo solo lectura:',len(registros),'elaboraciones; ',len(headers),'columnas')


## 4. Hallazgos y gráficos
Los gráficos se eligen por la relación que se quiere mostrar: ranking, distribución, comparación, correlación o evolución temporal. Las figuras llevan fuente, unidad y transformación. Los resultados que aparecen a continuación se calculan al ejecutar las celdas.

In [ ]:
X2=np.array([[*tasas(r),valor(r,'MATERIA GRASA'),valor(r,'PROTEINAS')] for r in registros],float)
X2s=normalizar(X2)
calificaciones={k:silhouette_score(X2s,KMeans(n_clusters=k,random_state=SEMILLA,n_init=10).fit_predict(X2s)) for k in range(2,7)}
k=max(calificaciones,key=calificaciones.get);grupos=KMeans(n_clusters=k,random_state=SEMILLA,n_init=10).fit_predict(X2s)
orden=sorted(calificaciones)
fig,ax=plt.subplots(figsize=(8,4));ax.plot(orden,[calificaciones[z] for z in orden],marker='o');ax.set_xlabel('Cantidad de grupos (k)');ax.set_ylabel('Silueta');grafico('Elección del número de grupos','índice de silueta','K-means, variables estandarizadas')
medias=np.array([np.nanmean(rend[grupos==g]) for g in range(k)]);cuentas=[int(np.sum(grupos==g)) for g in range(k)]
fig,ax=plt.subplots(figsize=(9,4));ax.bar([f'Grupo {g+1} (n={cuentas[g]})' for g in range(k)],medias*100);ax.set_ylabel('Rendimiento promedio (%)');grafico('Los grupos no rinden igual','porcentaje','promedio por grupo; rendimiento no usado para formar grupos')
print('Silueta:',{z:round(v,3) for z,v in calificaciones.items()});print('COMPRENSIÓN:',f'El mejor k es {k}; diferencia entre promedios extremos: {(max(medias)-min(medias))*100:.2f} puntos porcentuales.')


## 5. Comprensión: insight central
Tomá la frase que imprime la celda anterior como base de la conclusión. **No afirmes causas** que los datos no permitan demostrar. Explicá qué se observa, cuánto representa y qué limitaciones tiene.

## 6. Acción propuesta
Comparar recetas y prácticas de los grupos con peor rendimiento; el agrupamiento no prueba causalidad.

## 7. Control ético y límites
- Fuente: registro de producción suministrado por la empresa. Período: el cubierto por el Excel; sin recortar meses intencionalmente.
- Indicá unidades, cantidad de registros válidos y transformaciones aplicadas.
- Los gráficos de barras parten de cero; los gráficos de distribución y comparación no ocultan categorías.
- No se cambian ni completan los datos originales; cualquier imputación ocurre únicamente en el modelo.
- La interpretación se limita a asociaciones y predicciones, no a causalidad comprobada.
